In [ ]:
import os
import pandas as pd

def load_data(path: str, nrows: int = None) -> pd.DataFrame:
    transaction_path = os.path.join(path, "train_transaction.csv")
    identity_path = os.path.join(path, "train_identity.csv")

    if not os.path.exists(transaction_path):
        raise FileNotFoundError(f"Không tìm thấy file: {transaction_path}")

    tr = pd.read_csv(transaction_path, nrows=nrows)
    idt = pd.read_csv(identity_path, nrows=nrows)

    # bảng transaction left join với bảng identity theo TransactionID
    df = tr.merge(idt, on="TransactionID", how="left")

    print(f"[load_data] transaction: {tr.shape}, identity: {idt.shape}, merged: {df.shape}")
    if "isFraud" in df.columns:
        print(f"[load_data] Tỷ lệ gian lận: {df['isFraud'].mean():.4%}")
    return df


df = load_data(path="/kaggle/input/competitions/ieee-fraud-detection/", nrows=None)

: 

In [ ]:
print("Số dòng, số cột:", df.shape)
df.head()

In [ ]:
# kiểu data ở từng cột
dtype_summary = df.dtypes.value_counts()
print(dtype_summary)

In [ ]:
# thống kê số lượng và tỷ lệ phần trăm các giao dịch gian lận
fraud_count = df["isFraud"].value_counts()
fraud_pct = df["isFraud"].value_counts(normalize=True) * 100
print("Số lượng:\n", fraud_count)
print("\nTỷ lệ phần trăm:\n", fraud_pct)

In [ ]:
# biểu đồ cột so sánh số lượng giao dịch bình thường và gian lận
import matplotlib.pyplot as plt

fraud_count.plot(kind="bar", color=["steelblue", "salmon"])
plt.title("Phân bố giao dịch: Bình thường (0) vs Gian lận (1)")
plt.xlabel("isFraud")
plt.ylabel("Số lượng giao dịch")
plt.xticks(rotation=0)
plt.savefig("/kaggle/working/fig_fraud_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# thống kê tỉ lệ phần trăm các giá trị bị thiếu trong từng cột
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
print(missing_pct.head(20))

plt.figure(figsize=(8, 5))
missing_pct.plot(kind="hist", bins=30, color="steelblue")
plt.title("Phân bố % giá trị thiếu theo cột")
plt.xlabel("% giá trị thiếu")
plt.ylabel("Số lượng cột")
plt.savefig("/kaggle/working/fig_missing_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## Mục 1: Mô tả bài toán

**Lĩnh vực:** Phát hiện gian lận tài chính (Financial Fraud Detection) trong
giao dịch thương mại điện tử.

**Nguồn dữ liệu:** Bộ dữ liệu IEEE-CIS Fraud Detection, do công ty Vesta
Corporation cung cấp thông qua cuộc thi Kaggle (2019). Sau khi merge 2 file
`train_transaction.csv` và `train_identity.csv` theo `TransactionID`, tập
dữ liệu gồm **590,540 giao dịch** với **434 đặc trưng** (399 cột số, 31 cột
phân loại, 4 cột số nguyên).

**Vấn đề:** Dữ liệu mất cân bằng lớp (class imbalance) rất nghiêm trọng —
chỉ **20,663 giao dịch (3.50%)** là gian lận, còn lại **569,877 giao dịch
(96.50%)** là bình thường. Đây là thách thức cốt lõi: một mô hình đoán bừa
"không gian lận" cho mọi giao dịch vẫn đạt 96.5% accuracy, nên Accuracy
không phải là thước đo phù hợp cho bài toán này — cần dùng Precision,
Recall, F1-score và PR-AUC.

Ngoài ra, dữ liệu có mức độ thiếu giá trị (missing value) không đồng đều:
một nhóm lớn khoảng 130 cột (chủ yếu các đặc trưng ẩn danh `V1`-`V339`)
thiếu khoảng 75-80%, và một nhóm khác (`id_*`, `D*`, `dist2`) thiếu tới
85-99%. Việc xử lý hợp lý nhóm dữ liệu thiếu này là trọng tâm của bước
tiền xử lý.

**Mục tiêu khai phá dữ liệu:** Xây dựng mô hình phân loại (Classification)
dự đoán xác suất một giao dịch là gian lận, kết hợp với kỹ thuật không
giám sát (Clustering/Anomaly Detection) để phát hiện nhóm giao dịch bất
thường không dựa vào nhãn có sẵn.